In [ ]:
%mkdir /kaggle/temp/

In [ ]:
! conda install -c conda-forge gdcm -y;

In [ ]:
!pip install wandb --upgrade

In [ ]:
# %%capture
! pip install pytorch-metric-learning
! pip install timm
! pip install pytorch-lightning
! pip install lightning-bolts

# Ideas to try

https://www.kaggle.com/rerere/covid-19-image-enhancement-in-progress

# Load libraries

In [ ]:
import platform
import numpy as np 
import pandas as pd 
import os
from tqdm.notebook import tqdm
import cv2
import pydicom
import random
import gdcm
import glob
import gc
from math import ceil
import matplotlib.pyplot as plt
from pydicom.pixel_data_handlers.util import apply_voi_lut
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score

import albumentations

# from sklearn.model_selection import StratifiedKFold, GroupKFold

import math
import psutil

DATA_PATH = '/kaggle/input/siim-covid19-detection/'

In [ ]:
from PIL import Image

import torch
torch.manual_seed(0)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

import timm

import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from pytorch_metric_learning import losses
from torch.optim import lr_scheduler

import warnings
warnings.simplefilter('ignore')

In [ ]:
import pytorch_lightning as pl
import wandb
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.metrics.functional import accuracy
from pytorch_lightning.callbacks import LearningRateMonitor
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.callbacks import ModelCheckpoint

from torch.optim import Adam
from torch.nn.functional import cross_entropy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pl.seed_everything(101)

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

wandb_api = user_secrets.get_secret("wandb") 

wandb.login(key=wandb_api)

In [ ]:
import torch
torch.cuda.get_device_properties(0).total_memory

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Configs

In [ ]:
Config = dict (
    train_pcent = 0.85,
    model_name = 'resnet50',
    image_size = (400, 400),
    num_classes = 4,
    batch_size = 32,
    num_workers = 8,
    NB_EPOCHS = 30,
    scaler = GradScaler(),
    scheduler = 'CosineAnnealingLR',
    weight_decay = 1e-6,
    T_max = 10,
    min_lr = 1e-6,
    lr = 1e-5
)

# Custom functions

In [ ]:
# Function to store pickle object
def pickle_dump(path, saveobj):
    import pickle
    filehandler = open(path,"wb")
    pickle.dump(saveobj,filehandler)
    print("File pickled")
    filehandler.close()

In [ ]:
# Function to load pickle object
def pickle_load(path):
    import pickle
    file = open(path,'rb')
    loadobj = pickle.load(file)
    file.close()
    return loadobj

In [ ]:
def dicom2array(path, voi_lut=True, fix_monochrome=True):
    dicom = pydicom.read_file(path)
    # VOI LUT (if available by DICOM device) is used to
    # transform raw DICOM data to "human-friendly" view
    if voi_lut:
        data = apply_voi_lut(dicom.pixel_array, dicom)
    else:
        data = dicom.pixel_array
    # depending on this value, X-ray may look inverted - fix that:
    if fix_monochrome and dicom.PhotometricInterpretation == "MONOCHROME1":
        data = np.amax(data) - data
    data = data - np.min(data)
    data = data / np.max(data)
    data = (data * 255).astype(np.uint8)
    return data

# Data Handling

In [ ]:
class SIIMData(Dataset):
    def __init__(self, df, is_train=True, augments=None, img_size=Config['image_size']):
        super().__init__()
        self.df = df.sample(frac=1).reset_index(drop=True)
        self.is_train = is_train
        self.augments = augments
        self.img_size = img_size
        
    def __getitem__(self, idx):
        image_id = self.df['StudyInstanceUID'].values[idx]
        
        image_path = self.df['path'].values[idx]
        image = dicom2array(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        image = cv2.resize(image, Config['image_size'])
        
#         image = image.astype(np.int)
        
        # Augments must be albumentations
        if self.augments:
            image = self.augments(image=image)['image']
        
        image = torch.tensor(image)
        
        if self.is_train:
            label = self.df[self.df['StudyInstanceUID'] == image_id].values.tolist()[0][3:7]#.index(1)
            return image, torch.tensor(label)
        
        return image
    
    def __len__(self):
        return len(self.df)

In [ ]:
transforms_train = albumentations.Compose([
#     albumentations.RandomCrop(width=256, height=256, p=1),
    albumentations.VerticalFlip(p=0.3),
    albumentations.HorizontalFlip(p=0.5),
    albumentations.Rotate(p=0.5),
#     albumentations.GaussianBlur(p=0.3),
#     albumentations.Affine(p=0.3),
    albumentations.RandomBrightnessContrast(p=0.2),
    albumentations.CoarseDropout(p=0.3),
    albumentations.Normalize()
])

transforms_valid = albumentations.Compose([
    albumentations.Normalize()
])

# Data loader testing

In [ ]:
# train_image = pd.read_csv("/kaggle/input/siim-covid19-detection/train_image_level.csv")
# train_study = pd.read_csv("/kaggle/input/siim-covid19-detection/train_study_level.csv")

# TRAIN_DIR = "/kaggle/input/siim-covid19-detection/train/"
# train_study['StudyInstanceUID'] = train_study['id'].apply(lambda x: x.replace('_study', ''))
# train = train_image.merge(train_study, on='StudyInstanceUID')

# # Make a path folder
# paths = []
# for instance_id in tqdm(train['StudyInstanceUID']):
#     paths.append(glob.glob(os.path.join(TRAIN_DIR, instance_id +"/*/*"))[0])

# train['path'] = paths

# train = train.drop(['id_x', 'id_y'], axis=1)

In [ ]:
# # Assign train/val datasets for use in dataloaders
# nb_training_samples = int(Config['train_pcent'] * train.path.shape[0])
# train_data = train[:nb_training_samples]
# valid_data = train[nb_training_samples:]

# print(f"[INFO] Training on {train_data.shape[0]} samples ({int(Config['train_pcent']*100)}%) and validation on {valid_data.shape[0]} ({ceil(abs(1 - Config['train_pcent']) * 100)}%) samples")

# training_set = SIIMData(
#                     df=train_data,
#                     augments=transforms_train
#                 )

# validation_set = SIIMData(
#                     df=valid_data,
#                     augments=transforms_valid
#                 )

In [ ]:
# train_loader = DataLoader(training_set, batch_size=32, shuffle=False, pin_memory=True)

In [ ]:
# for batch in train_loader:
#     x, y = batch
#     break

In [ ]:
# x[0].shape

In [ ]:
# plt.figure()
# fig, ax = plt.subplots(3, 2, figsize=[15, 8])

# def rand_batch_ind():
#     return round(round(random.random(),1)*31)


# for i in range(3):
#     for j in range(2):
#         ax[i][j].imshow(x[rand_batch_ind()])
        
# plt.show()

# Pytorch lightning

## Lit Data Module

In [ ]:
class SIIMCovid19DataModule(pl.LightningDataModule):
    def __init__(self, data_dir='./', batch_size=Config['batch_size'], num_classes=Config['num_classes']):

        super().__init__()

        self.data_dir = data_dir
        self.batch_size = batch_size

        # We hardcode dataset specific stuff here.
        self.num_classes = num_classes
#         self.dims = (3, 32, 32)
        self.transform = transforms.Compose([
            transforms.ToTensor()
        ])

    def prepare_data(self):
        train_image = pd.read_csv("/kaggle/input/siim-covid19-detection/train_image_level.csv")
        train_study = pd.read_csv("/kaggle/input/siim-covid19-detection/train_study_level.csv")
        
        TRAIN_DIR = "/kaggle/input/siim-covid19-detection/train/"
        train_study['StudyInstanceUID'] = train_study['id'].apply(lambda x: x.replace('_study', ''))
        self.train = train_image.merge(train_study, on='StudyInstanceUID')

        # Make a path folder
        paths = []
        for instance_id in tqdm(self.train['StudyInstanceUID']):
            paths.append(glob.glob(os.path.join(TRAIN_DIR, instance_id +"/*/*"))[0])

        self.train['path'] = paths

        self.train = self.train.drop(['id_x', 'id_y'], axis=1)

         
    def setup(self, stage=None):

        # Assign train/val datasets for use in dataloaders
        nb_training_samples = int(Config['train_pcent'] * self.train.path.shape[0])
        train_data = self.train[:nb_training_samples]
        valid_data = self.train[nb_training_samples:]

        print(f"[INFO] Training on {train_data.shape[0]} samples ({int(Config['train_pcent']*100)}%) and validation on {valid_data.shape[0]} ({ceil(abs(1 - Config['train_pcent']) * 100)}%) samples")
        
        self.training_set = SIIMData(
                            df=train_data,
                            augments=transforms_train
                        )

        self.validation_set = SIIMData(
                            df=valid_data,
                            augments=transforms_valid
                        )


    def train_dataloader(self):
        return DataLoader(self.training_set, batch_size=self.batch_size, shuffle=False, num_workers=8, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.validation_set, batch_size=self.batch_size, shuffle=False, num_workers=8, pin_memory=True)

## Lit Model

In [ ]:
class LitResNet50(pl.LightningModule):
    def __init__(self, model_name, num_classes, lr, weight_decay):
        
        super().__init__()
        self.save_hyperparameters()
        
        # Model architecture
        self.backbone = models.resnet50(pretrained=True)
        self.finetune_layer = nn.Linear(self.backbone.fc.out_features, self.hparams.num_classes)
        
        # compute the accuracy -- no need to roll your own!
        self.train_acc = pl.metrics.Accuracy()
        self.valid_acc = pl.metrics.Accuracy()
        
            
    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config['T_max'], eta_min=Config['min_lr'])
        
        lr_scheduler_lit = {
            'scheduler': scheduler,
            'name': 'CosineAnnealingLR'
        }

        return [optimizer], [lr_scheduler_lit]

    
    def forward(self, x, y=None):
        
        x = x.to(self.device).float()
        x = x.permute(0, 3, 1, 2)

        features = self.backbone(x)
        preds = self.finetune_layer(features)

        return preds

    
    def training_step(self, batch, batch_idx):
        
#         print("Training step starts")
        
        x, y = batch
        
        x = x.to(self.device).float()
        y = y.to(self.device).float()
        x = x.permute(0, 3, 1, 2)

        features = self.backbone(x)
        preds = self.finetune_layer(features)
        
        BCEloss = criterion1(preds, y)
        # Create logs
        self.log('train/BCEloss', BCEloss)
        
        preds = nn.functional.log_softmax(preds)
        
#         print("Training step ends")
        
        return BCEloss#, preds, y
    
        
    
    def validation_step(self, batch, batch_idx):
        
#         print("Validation step starts")
        
        x, y = batch
        
        x = x.to(self.device).float()
        y = y.to(self.device).float()
        x = x.permute(0, 3, 1, 2)

        features = self.backbone(x)
        preds = self.finetune_layer(features)
        
        BCEloss = criterion1(preds, y)
        # Create logs
        self.log('val/BCEloss', BCEloss)
        
        preds = nn.functional.log_softmax(preds)
        
#         print("Validation step ends")

        return BCEloss, preds, y
        
    
    def validation_epoch_end(self, val_step_outputs):
        """Called when the validation epoch ends."""
        
#         print("Validation epoch end starts")
        
        epoch = self.trainer.current_epoch
        
        preds = []
        y_true = []
        for out in val_step_outputs:
            _, pred, y = out
            preds.extend(pred.cpu().numpy())
            y_true.extend(y.cpu().numpy())
            
        
        # Calculate mAP score
        mAPloss = average_precision_score(y_true, preds)
        # Create logs
        self.log('val/mAP', mAPloss)
        
        # Histogram of logits
        self.logger.experiment.log(
        {"val/logits": wandb.Histogram(preds),
         "global_step": self.global_step})
        
        
        # Save latest model checkpoint
        print(f"Saving latest model checkpoint at epoch {epoch}; mAP score: {mAPloss:.4f}")
        filename = f'/kaggle/temp/resnet50-siim-covid19-{epoch}-{mAPloss:.4f}.ckpt'
        trainer.save_checkpoint(filename)
        
        # Save model as Model Artifact
        artifact = wandb.Artifact(name=f'resnet50-{epoch}-{mAPloss:.4f}', type='model')
        artifact.add_file(filename)
        run.log_artifact(artifact)
        
#         print("Validation epoch end ends")
        
            
    
    def on_save_checkpoint(self, checkpoint):
        pass
            
    
    def predict(self, batch, batch_idx):
        x = batch
        output = self(x, y=None)
        return output

In [ ]:
checkpoint_callback = ModelCheckpoint(
    monitor='val/mAP',
    dirpath='/kaggle/temp/',
    filename='resnet50-siim-covid19-{epoch:02d}-{train_loss:.4f}-{val_loss:.4f}',
    save_top_k=5,
    mode='max',
)

early_stop_callback = EarlyStopping(
   monitor='val/mAP',
   min_delta=0.00,
   patience=5,
   verbose=True,
   mode='max'
)

In [ ]:
lr_monitor = LearningRateMonitor(logging_interval='step')

# criterion1 = nn.CrossEntropyLoss()
criterion1 = nn.BCEWithLogitsLoss()

# criterion2 = sklearn.metrics.average_precision_score()

In [ ]:
classifier = LitResNet50(model_name = Config['model_name'], 
                         num_classes = Config['num_classes'], 
                         lr = Config['lr'], 
                         weight_decay = Config['weight_decay'])

## Training

In [ ]:
project = 'kaggle-siim-covid'
group = 'Study Classification'
experiment = 'ResNet50_400px_baseline'
notes = 'ResNet50 baseline run; 400px images; with transformations; BCE, mAP'

# Initialize W&B run
run = wandb.init(project=project, 
                 config=Config,
                 group=group, 
                 job_type='train',
                 tags=['train', 'resnet50'],
                 notes=notes)

In [ ]:
# Define a WandbLogger
wandb_logger = WandbLogger(project=project,
                           group=group,
#                            experiment=experiment,
#                            id=None,
                           notes=notes)

In [ ]:
# Data module
dm = SIIMCovid19DataModule()
# Define trainer
trainer = pl.Trainer(progress_bar_refresh_rate=5, gpus=1, 
                     max_epochs=50, 
                     check_val_every_n_epoch = 2,
#                      limit_train_batches=5, 
#                      limit_val_batches=3,
                     stochastic_weight_avg=True, 
                     accumulate_grad_batches=2,
                     callbacks=[checkpoint_callback, lr_monitor],
                     logger=wandb_logger,
#                      fast_dev_run=True
                    ) 
# Fit model
trainer.fit(classifier, datamodule=dm)

In [ ]:
!ls /kaggle

In [ ]:
!cp /kaggle/temp/*.ckpt /kaggle/working/